# Market Risk Core: VaR and ES Formula Validation

**Purpose:** Validate the exact lognormal (GBM) VaR and ES formulas for long and short positions against golden-fixture values from the MATH 5320 formula sheet. Also compares with the normal-linear approximation and illustrates how confidence levels interact.

In [ ]:
%matplotlib inline
import sys, math
sys.path.insert(0, "..")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from src.risk.lognormal import (
    var_long_lognormal, es_long_lognormal,
    var_short_lognormal, es_short_lognormal,
)
from src.risk.normal import normal_var, normal_es, portfolio_delta_normal_mean_var
print('Imports OK')

## Section 2 — Long GBM VaR/ES (golden fixture)

In [ ]:
V0, mu, sigma, h = 10_000.0, 0.02, 0.20, 1.0
# mu = arithmetic drift; m = log-return drift = mu - 0.5*sigma^2 = 0.00
# For this fixture mu = 0.02 → m = 0.02 - 0.5*(0.04) = 0.00 (symmetric)
long_var_99  = var_long_lognormal(V0, mu, sigma, h, 0.99)
long_es_975  = es_long_lognormal(V0, mu, sigma, h, 0.975)
print(f'Long VaR 99%:  {long_var_99:,.2f}  (expected ≈ 3,720)')
print(f'Long ES  97.5%: {long_es_975:,.2f} (expected ≈ 4,337)')

## Section 3 — Short GBM VaR/ES

In [ ]:
short_var_99  = var_short_lognormal(V0, mu, sigma, h, 0.99)
short_es_975  = es_short_lognormal(V0, mu, sigma, h, 0.975)
print(f'Short VaR 99%:  {short_var_99:,.2f}  (expected ≈ 5,924)')
print(f'Short ES  97.5%: {short_es_975:,.2f} (expected ≈ 5,999)')

## Section 4 — Normal Linear VaR/ES (benchmark comparison)

In [ ]:
mean_pnl, std_pnl = 0.0, V0 * sigma  # simplified: μ=0, σ≈2000
nvar = normal_var(mean_pnl, std_pnl, 0.99)
nes  = normal_es(mean_pnl, std_pnl, 0.975)
print(f'Normal VaR 99%:  {nvar:,.2f}')
print(f'Normal ES  97.5%: {nes:,.2f}')
print(f'(Normal is symmetric; lognormal long-VaR is smaller due to bounded downside)')

## Section 5 — Mixed Confidence Levels

The course uses **mixed** confidence levels in several homework problems:
- **HW VI**: 99% VaR / 97.5% ES
- **HW VIII**: 95% VaR / 97.5% ES

VaR and ES need not use the same confidence level. ES is always a tail conditional expectation, so `ES(q) >= VaR(q)` for any common q.

In [ ]:
configs = [(0.99, 0.975), (0.99, 0.99), (0.95, 0.975)]
print(f'{'VaR conf':>10} {'ES conf':>10} {'VaR($)':>12} {'ES($)':>12}')
print('-' * 46)
for vc, ec in configs:
    v = var_long_lognormal(V0, mu, sigma, h, vc)
    e = es_long_lognormal(V0, mu, sigma, h, ec)
    print(f'{vc:>10.0%} {ec:>10.1%} {v:>12,.0f} {e:>12,.0f}')

## Section 6 — VaR vs ES as Function of Confidence (plot)

In [ ]:
confs = np.linspace(0.90, 0.999, 200)
vars_ = [var_long_lognormal(V0, mu, sigma, h, c) for c in confs]
ess_  = [es_long_lognormal(V0, mu, sigma, h, c) for c in confs]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(confs * 100, vars_, label='VaR (long, GBM)', color='steelblue')
ax.plot(confs * 100, ess_,  label='ES  (long, GBM)', color='firebrick', linestyle='--')
ax.fill_between(confs * 100, vars_, ess_, alpha=0.12, color='firebrick')
ax.set_xlabel('Confidence Level (%)')
ax.set_ylabel('Risk Measure ($)')
ax.set_title('VaR and ES vs Confidence Level (Long GBM, V0=10k, σ=20%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('ES > VaR at every confidence level — by construction of conditional expectation.')

## Section 7 — Short vs Long Comparison (bar chart)

In [ ]:
labels = ['Long VaR 99%', 'Long ES 97.5%', 'Short VaR 99%', 'Short ES 97.5%']
values = [long_var_99, long_es_975, short_var_99, short_es_975]
colors = ['steelblue', 'skyblue', 'firebrick', 'salmon']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, values, color=colors, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Dollar Risk ($)')
ax.set_title('Long vs Short GBM Risk Measures (V0=10k, μ=2%, σ=20%, h=1yr)')
ax.text(0.98, 0.92, 'Short VaR > Long VaR: short position\nloses when price rises (opposite tail)',
        transform=ax.transAxes, ha='right', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Section 8 — Validation Summary

| Metric | Value | Notes |
|---|---|---|
| Long VaR 99% | ≈ 3,720 | LN01 course fixture |
| Long ES 97.5% | ≈ 4,337 | ES uses 97.5%, not 99% |
| Short VaR 99% | ≈ 5,924 | Short position has heavier upper tail |
| Short ES 97.5% | ≈ 5,999 | ES > VaR also for short |
| ES > VaR | Always | By definition of conditional expectation |
| Normal ES 97.5% | computed above | Normal is symmetric; GBM has positive skew |

**Key insight:** For a short position, the loss occurs when the asset price *rises*. Because asset prices are lognormally distributed (unbounded above), the short-position tail is heavier than the long-position tail.